In [ ]:
#This file is based on FootNet's implementation of identifying a strikefoot. 
#A few limitations to address are that FootNet's research lab had 33 keypoints, obtained from motion captrue data whereas YOLOv8's pose models only offer 17 (half that)
#Another limitation lies in the fact that the lab had forceplates to recognise when strikefoot's took place (on training data). I will have to comb through the frames and manually see where strikefoot takes place
import cv2
import json
from collections import defaultdict

video_dir = "/Users/abhinavarora/Desktop/CadenceCV/Videos/Normal Running.MOV"
video_num = "Video 1"

final = defaultdict()

#This function will go through the video frame by frame. Each frame as classified as "u" = left foot up, "d" = left foot down, "s" = skip (if its neither).
#"u" and "d" apply to the EXACT moment the foot goes up or down respectively. Each consecutive pair of "u" and "d" will be classified as a gait cycle
def extract_gait_cycle():
    gait_cycles = []
    single_gait_cycle = []
    cap = cv2.VideoCapture(video_dir)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for i in range(frame_count):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow('Frame', frame)
        key = cv2.waitKey(0)
        #Constructing a single gait cycle and appending them
        if key == ord('u'):
            single_gait_cycle.append(i)
        elif key == ord('d'):
            single_gait_cycle.append(i)
            gait_cycles.append(single_gait_cycle)
            single_gait_cycle = []
        elif key == ord('c'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    return gait_cycles

#Manual labelling done via tracking the left foot
left_foot_cycle = extract_gait_cycle()
gait_frames = {"Gait Cycle": left_foot_cycle}
final = {video_num: gait_frames}

In [ ]:
#Saving to a JSON file so gait cycle frames can be loaded as parameters for the LSTM
with open("strikefoot_frames.json", "a") as file:
    json.dump(final, file)